# 04 — Error Analysis

Inspect false positives / negatives and compare slices without exposing customer information.

**Data:** Synthetic invoice and purchase-order records. No production data. Not a bank credit scoring model.

In [ ]:
from pathlib import Path
import sys
root = Path.cwd().resolve()
if root.name == "notebooks": root = root.parent
sys.path.insert(0, str(root / "src"))

In [ ]:
from proofflow.ml import synthetic_invoices, split_data, build_pipeline
from sklearn.metrics import precision_recall_fscore_support
import pandas as pd
x_train,x_test,y_train,y_test=split_data(synthetic_invoices())
model=build_pipeline().fit(x_train,y_train)
results=x_test.copy()
results["actual"] = y_test
results["score"] = model.predict_proba(x_test)[:,1]
results["predicted"] = (results.score>=0.5).astype(int)
results["error"] = results.actual.ne(results.predicted)
print("Error count:",int(results.error.sum()))
print(results.loc[results.error].sort_values("score").head(12).to_string(index=False))

In [ ]:
for currency, section in results.groupby("currency"):
    pr,re,f1,_=precision_recall_fscore_support(section.actual,section.predicted,average="binary",zero_division=0)
    print(f"{currency}: n={len(section)} precision={pr:.3f} recall={re:.3f} f1={f1:.3f}")

**Limitations:** Currency slices are randomly generated; differences here do not establish real demographic or business fairness. Human approval is still required for actual operations.